In [1]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt

# Load CSV into a SQLite database
df = pd.read_csv('creditcard.csv')
df['Hour'] = (df['Time'] / 3600) % 24
df['Amount_Bracket'] = pd.cut(df['Amount'],
    bins=[0, 50, 100, 200, 500, 1000, 5000, 30000],
    labels=['$0-50', '$50-100', '$100-200', '$200-500', '$500-1K', '$1K-5K', '$5K+'])

# Create SQLite database
conn = sqlite3.connect('fraud_analysis.db')
df.to_sql('transactions', conn, if_exists='replace', index=False)

print("Database created successfully!")
print(f"Table 'transactions' loaded with {len(df):,} rows")

# Verify
result = pd.read_sql("SELECT COUNT(*) as total, SUM(Class) as fraud_count FROM transactions", conn)
print(result)


Database created successfully!
Table 'transactions' loaded with 284,807 rows
    total  fraud_count
0  284807          492


In [2]:
# Query 1: Overall fraud summary KPIs
query1 = """
SELECT 
    COUNT(*) as total_transactions,
    SUM(Class) as total_fraud,
    COUNT(*) - SUM(Class) as total_legitimate,
    ROUND(AVG(Class) * 100, 4) as fraud_rate_pct,
    ROUND(SUM(CASE WHEN Class=1 THEN Amount ELSE 0 END), 2) as total_fraud_loss,
    ROUND(AVG(CASE WHEN Class=1 THEN Amount END), 2) as avg_fraud_amount,
    ROUND(AVG(CASE WHEN Class=0 THEN Amount END), 2) as avg_legit_amount,
    ROUND(MAX(CASE WHEN Class=1 THEN Amount END), 2) as max_fraud_amount
FROM transactions
"""

result1 = pd.read_sql(query1, conn)
print("Query 1: Overall Fraud KPIs")
print("="*50)
for col in result1.columns:
    print(f"{col:25} : {result1[col].values[0]:,}")

Query 1: Overall Fraud KPIs
total_transactions        : 284,807
total_fraud               : 492
total_legitimate          : 284,315
fraud_rate_pct            : 0.1727
total_fraud_loss          : 60,127.97
avg_fraud_amount          : 122.21
avg_legit_amount          : 88.29
max_fraud_amount          : 2,125.87


In [3]:
# Query 2: Fraud rate by amount bracket
query2 = """
SELECT 
    Amount_Bracket,
    COUNT(*) as total_transactions,
    SUM(Class) as fraud_count,
    ROUND(AVG(Class) * 100, 4) as fraud_rate_pct,
    ROUND(SUM(CASE WHEN Class=1 THEN Amount ELSE 0 END), 2) as fraud_loss,
    ROUND(AVG(CASE WHEN Class=1 THEN Amount END), 2) as avg_fraud_amount
FROM transactions
GROUP BY Amount_Bracket
ORDER BY fraud_rate_pct DESC
"""

result2 = pd.read_sql(query2, conn)
print("Query 2: Fraud Rate by Amount Bracket")
print("="*50)
print(result2.to_string(index=False))

Query 2: Fraud Rate by Amount Bracket
Amount_Bracket  total_transactions  fraud_count  fraud_rate_pct  fraud_loss  avg_fraud_amount
           NaN                1825           27          1.4795        0.00              0.00
       $500-1K                6202           26          0.4192    17957.57            690.68
        $1K-5K                2885            9          0.3120    13237.33           1470.81
      $200-500               19695           50          0.2539    15770.99            315.42
      $100-200               27671           45          0.1626     6142.82            136.51
       $50-100               37254           56          0.1503     4945.33             88.31
         $0-50              189220          279          0.1474     2073.93              7.43
          $5K+                  55            0          0.0000        0.00               NaN


In [7]:
query3 = """
select
CAST(Hour as INT) as hour_of_the_day,
count(*) as Total_transactions,
sum(Class) as fraud_count,
Round(Avg(Class)*100,4) as fraud_rate_percentage
from transactions
group by Cast(Hour as INT)
order by fraud_rate_percentage desc
limit 10
"""

result3 = pd.read_sql(query3, conn)
print("Query 3: Top 10 fruad rate by hours")
print('='*50)
print(result3.to_string(index=False))

Query 3: Top 10 fruad rate by hours
 hour_of_the_day  Total_transactions  fraud_count  fraud_rate_percentage
               2                3328           57                 1.7127
               4                2209           23                 1.0412
               3                3492           17                 0.4868
               5                2990           11                 0.3679
               7                7243           23                 0.3175
              11               16856           53                 0.3144
               1                4220           10                 0.2370
               6                4101            9                 0.2195
              18               17039           33                 0.1937
              23               10938           21                 0.1920


In [10]:
query4 = """
select
case 
when cast(Hour as Int) Between 0 and 5 then 'Night 12AM-6AM'
when cast(Hour as Int) Between 6 and 11 then 'Morning 6AM-11AM'
when cast(hour as Int) Between 12 and 17 then 'Noon 11AM-6PM'
else 'Evening 7PM-12AM'
end as time_period,
count(*) as Total_transactions,
sum(class) as fraud_count,
round(avg(Class)*100,4) as fraud_rate_pct,
round(sum(case when class=1 then Amount else 0 end),2) as fraud_loss
from transactions
group by time_period
order by fraud_rate_pct desc

"""
result4 = pd.read_sql(query4, conn)
print("Query 4: Fraud Analysis by Time Period")
print('='*50)
print(result4.to_string(index=False))


Query 4: Fraud Analysis by Time Period
     time_period  Total_transactions  fraud_count  fraud_rate_pct  fraud_loss
  Night 12AM-6AM               23934          124          0.5181    10816.15
Morning 6AM-11AM               70912          118          0.1664    14371.84
   Noon 11AM-6PM               96435          134          0.1390    19355.18
Evening 7PM-12AM               93526          116          0.1240    15584.80


In [11]:
query5 = """
SELECT
    Amount,
    CAST(Hour as INT) as hour,
    CASE
        WHEN CAST(Hour as INT) BETWEEN 0 AND 5 THEN 'Night 12AM-6AM'
        WHEN CAST(Hour as INT) BETWEEN 6 AND 11 THEN 'Morning 6AM-11AM'
        WHEN CAST(Hour as INT) BETWEEN 12 AND 17 THEN 'Noon 11AM-6PM'
        ELSE 'Evening 7PM-12AM'
    END as time_period
FROM transactions
WHERE Class = 1
ORDER BY Amount DESC
LIMIT 10
"""
result5 = pd.read_sql(query5, conn)
print("Query 5: Top 10 Highest Value Fraud Transactions")
print("="*50)
print(result5.to_string(index=False))

Query 5: Top 10 Highest Value Fraud Transactions
 Amount  hour      time_period
2125.87    10 Morning 6AM-11AM
1809.68     2   Night 12AM-6AM
1504.93    18 Evening 7PM-12AM
1402.16    17    Noon 11AM-6PM
1389.56    16    Noon 11AM-6PM
1354.25    18 Evening 7PM-12AM
1335.00    12    Noon 11AM-6PM
1218.89     5   Night 12AM-6AM
1096.99    18 Evening 7PM-12AM
 996.27    16    Noon 11AM-6PM


In [12]:
query6 = """
SELECT
    Amount,
    CAST(Hour as INT) as hour,
    SUM(Amount) OVER (ORDER BY Time) as cumulative_loss
FROM transactions
WHERE Class = 1
ORDER BY Time
LIMIT 15
"""
result6 = pd.read_sql(query6, conn)
print("Query 6: Running Total of Fraud Loss")
print("="*50)
print(result6.to_string(index=False))

Query 6: Running Total of Fraud Loss
 Amount  hour  cumulative_loss
   0.00     0             0.00
 529.00     0           529.00
 239.93     1           768.93
  59.00     1           827.93
   1.00     2           828.93
   1.00     2           829.93
   1.00     2           830.93
   1.00     2           831.93
   1.00     2           832.93
   1.00     2           833.93
   1.00     2           834.93
   1.00     2           835.93
   1.00     2           836.93
   1.00     2           837.93
   1.00     2           838.93


In [13]:
query7 = """
SELECT
    Amount_Bracket,
    COUNT(*) as total_transactions,
    SUM(Class) as fraud_count,
    ROUND(AVG(Class)*100, 4) as fraud_rate,
    ROUND(SUM(CASE WHEN Class=1 THEN Amount ELSE 0 END), 2) as fraud_loss,
    CASE
        WHEN ROUND(AVG(Class)*100, 4) > 0.3 THEN 'HIGH RISK'
        WHEN ROUND(AVG(Class)*100, 4) BETWEEN 0.15 AND 0.3 THEN 'MEDIUM RISK'
        ELSE 'LOW RISK'
    END as risk_flag
FROM transactions
GROUP BY Amount_Bracket
ORDER BY fraud_rate DESC
"""
result7 = pd.read_sql(query7, conn)
print("Query 7: Fraud Rate by Amount Bracket with Risk Flag")
print("="*55)
print(result7.to_string(index=False))

Query 7: Fraud Rate by Amount Bracket with Risk Flag
Amount_Bracket  total_transactions  fraud_count  fraud_rate  fraud_loss   risk_flag
           NaN                1825           27      1.4795        0.00   HIGH RISK
       $500-1K                6202           26      0.4192    17957.57   HIGH RISK
        $1K-5K                2885            9      0.3120    13237.33   HIGH RISK
      $200-500               19695           50      0.2539    15770.99 MEDIUM RISK
      $100-200               27671           45      0.1626     6142.82 MEDIUM RISK
       $50-100               37254           56      0.1503     4945.33 MEDIUM RISK
         $0-50              189220          279      0.1474     2073.93    LOW RISK
          $5K+                  55            0      0.0000        0.00    LOW RISK


In [14]:
query8 = """
SELECT 
    Amount,
    CAST(Hour as INT) as hour,
    CASE
        WHEN CAST(Hour as INT) BETWEEN 0 AND 5 THEN 'Night'
        WHEN CAST(Hour as INT) BETWEEN 6 AND 11 THEN 'Morning'
        WHEN CAST(Hour as INT) BETWEEN 12 AND 17 THEN 'Noon'
        ELSE 'Evening'
    END as time_period,
    ROUND(Amount - (SELECT AVG(Amount) FROM transactions WHERE Class=1), 2) as above_avg_by
FROM transactions
WHERE Class = 1
AND Amount > (SELECT AVG(Amount) FROM transactions WHERE Class=1)
ORDER BY Amount DESC
LIMIT 15
"""
result8 = pd.read_sql(query8, conn)
print("Query 8: Fraud Transactions Above Average Amount")
print("="*55)
print(result8.to_string(index=False))

Query 8: Fraud Transactions Above Average Amount
 Amount  hour time_period  above_avg_by
2125.87    10     Morning       2003.66
1809.68     2       Night       1687.47
1504.93    18     Evening       1382.72
1402.16    17        Noon       1279.95
1389.56    16        Noon       1267.35
1354.25    18     Evening       1232.04
1335.00    12        Noon       1212.79
1218.89     5       Night       1096.68
1096.99    18     Evening        974.78
 996.27    16        Noon        874.06
 925.31    13        Noon        803.10
 829.41     0       Night        707.20
 824.83    19     Evening        702.62
 802.52    11     Morning        680.31
 776.83    11     Morning        654.62


In [15]:
query9 = """
SELECT
    time_period,
    SUM(CASE WHEN Amount_Bracket = '$0-50'   THEN 1 ELSE 0 END) as '$0_50',
    SUM(CASE WHEN Amount_Bracket = '$50-100' THEN 1 ELSE 0 END) as '$50_100',
    SUM(CASE WHEN Amount_Bracket = '$100-200'THEN 1 ELSE 0 END) as '$100_200',
    SUM(CASE WHEN Amount_Bracket = '$200-500'THEN 1 ELSE 0 END) as '$200_500',
    SUM(CASE WHEN Amount_Bracket = '$500-1K' THEN 1 ELSE 0 END) as '$500_1K',
    SUM(CASE WHEN Amount_Bracket = '$1K-5K'  THEN 1 ELSE 0 END) as '$1K_5K'
FROM (
    SELECT
        Amount_Bracket,
        Class,
        CASE
            WHEN CAST(Hour as INT) BETWEEN 0 AND 5  THEN 'Night'
            WHEN CAST(Hour as INT) BETWEEN 6 AND 11 THEN 'Morning'
            WHEN CAST(Hour as INT) BETWEEN 12 AND 17 THEN 'Noon'
            ELSE 'Evening'
        END as time_period
    FROM transactions
    WHERE Class = 1
) 
GROUP BY time_period
ORDER BY time_period
"""
result9 = pd.read_sql(query9, conn)
print("Query 9: Fraud Count Matrix — Time Period vs Amount Bracket")
print("="*65)
print(result9.to_string(index=False))

Query 9: Fraud Count Matrix — Time Period vs Amount Bracket
time_period  $0_50  $50_100  $100_200  $200_500  $500_1K  $1K_5K
    Evening     65        9         7        21        5       3
    Morning     52       33        15         5        7       1
      Night     89        6         9         7        5       2
       Noon     73        8        14        17        9       3


In [17]:
query10 = """
SELECT 'Total Transactions' as metric, CAST(COUNT(*) as TEXT) as value 
FROM transactions
UNION ALL
SELECT 'Total Fraud Cases', CAST(SUM(Class) as TEXT)
FROM transactions
UNION ALL
SELECT 'Overall Fraud Rate', CAST(ROUND(AVG(Class)*100, 4) as TEXT) || '%'
FROM transactions
UNION ALL
SELECT 'Total Fraud Loss', '$' || CAST(ROUND(SUM(CASE WHEN Class=1 THEN Amount ELSE 0 END), 2) as TEXT)
FROM transactions
UNION ALL
SELECT 'Avg Fraud Amount', '$' || CAST(ROUND(AVG(CASE WHEN Class=1 THEN Amount END), 2) as TEXT)
FROM transactions
UNION ALL
SELECT 'Avg Legit Amount', '$' || CAST(ROUND(AVG(CASE WHEN Class=0 THEN Amount END), 2) as TEXT)
FROM transactions
UNION ALL
SELECT 'Peak Fraud Hour', '2AM (1.71% fraud rate)' FROM (SELECT 1)
UNION ALL
SELECT 'Highest Risk Bracket', '$500-$1K (0.42% fraud rate)' FROM (SELECT 1)
UNION ALL
SELECT 'Highest Risk Period', 'Night 12AM-6AM (0.52% fraud rate)' FROM (SELECT 1)
UNION ALL
SELECT 'Card Testing Cases', CAST(SUM(CASE WHEN Class=1 AND Amount <= 1 THEN 1 ELSE 0 END) as TEXT) || ' transactions under $1'
FROM transactions
"""
result10 = pd.read_sql(query10, conn)
print("Query 10: Executive KPI Summary")
print("="*45)
print(result10.to_string(index=False))

Query 10: Executive KPI Summary
              metric                             value
  Total Transactions                            284807
   Total Fraud Cases                               492
  Overall Fraud Rate                           0.1727%
    Total Fraud Loss                         $60127.97
    Avg Fraud Amount                           $122.21
    Avg Legit Amount                            $88.29
     Peak Fraud Hour            2AM (1.71% fraud rate)
Highest Risk Bracket       $500-$1K (0.42% fraud rate)
 Highest Risk Period Night 12AM-6AM (0.52% fraud rate)
  Card Testing Cases         181 transactions under $1
